In [1]:
%pip cache purge

%pip install -r ../requirements.txt


Files removed: 0 (0 bytes)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil

DIRECTORIES = [
    "../models", 
    "../data/raw/files"
]
deleted_files = []

for directory in DIRECTORIES:
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        continue

    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)

        if item == ".gitkeep":
            continue  # Skip .gitkeep

        if os.path.isfile(item_path):
            os.remove(item_path)
            deleted_files.append(item_path)
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            deleted_files.append(item_path + "/")  

if deleted_files:
    print("Deleted files and directories:")
    for file in deleted_files:
        print(f" - {file}")
else:
    print("No files to delete.")


Deleted files and directories:
 - ../models/eeg_model_motor_imagery_left_right_20250315_103457.joblib
 - ../models/pipeline_comparison.png
 - ../models/eeg_model_motor_imagery_left_right_20250315_103457_info.json
 - ../models/confusion_matrices.png
 - ../models/eeg_dataset_20250315_102542/
 - ../models/subject_performances.png
 - ../models/prediction_results/
 - ../data/raw/files/MNE-eegbci-data/


In [ ]:
# Crear los directorios necesarios
import os
os.makedirs('../models', exist_ok=True)

# Establecer semilla aleatoria para reproducibilidad
import numpy as np
import random
RANDOM_SEED = None
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
# Importar funciones de nuestros scripts
from preprocessing import load_subjects_for_experiment, normalize_labels, save_metadata, save_processed_data, clean_memory
from pipeline import compare_pipelines, hold_one_out_experiment, train_and_save_model, plot_subject_performances, plot_confusion_matrices, pipeline_comparison_chart
from predict import load_model, load_specific_subject, predict_eeg, visualize_predictions_over_time


In [ ]:
# Configuración
NUM_SUBJECTS = 3  # Número de sujetos a cargar
RANDOM_SEED = None   # Semilla para reproducibilidad

print(f"Descargando y preprocesando datos EEG de {NUM_SUBJECTS} sujetos aleatorios...")

# Descargar y preprocesar, eligiendo un grupo de experimento aleatorio
eeg_data = load_subjects_for_experiment(
    num_subjects=NUM_SUBJECTS, 
    experiment_group=None,  # None para selección aleatoria
    random_seed=RANDOM_SEED
)

# Normalizar etiquetas para asegurar compatibilidad entre diferentes paradigmas
eeg_data = normalize_labels(eeg_data)

# Resumen de los datos cargados
print("\nResumen de los datos EEG cargados:")
for i, info in enumerate(eeg_data):
    print(f"Sujeto {i+1}: ID={info['subject']}, Tarea={info['task_type']}, Paradigma={info['paradigm']}")
    print(f"  Forma de datos: {info['X'].shape}, Clases: {list(info['event_id'].keys())}")
    print(f"  Distribución de clases: {info['class_counts']}")
    print()

# Guardar datos procesados con metadatos completos
saved_info = save_processed_data(eeg_data)
print(f"\nDatos guardados en: {saved_info['dataset_dir']}")

# Imprimir la estructura para diagnosticar
print("Estructura de saved_info['config']:")
for key in saved_info['config']:
    print(f"  - {key}")

# Usar las claves correctas para acceder a la información
n_samples = saved_info['config'].get('n_samples') or saved_info['config'].get('dataset_info', {}).get('n_samples')
experiment_group = saved_info['config'].get('experiment_group') or saved_info['config'].get('dataset_info', {}).get('experiment_group')

print(f"Total de muestras: {n_samples}")
print(f"Grupo de experimento: {experiment_group}")

# Limpiar memoria
light_data = clean_memory(eeg_data)
print("Memoria limpiada: se eliminaron los arrays grandes de X e y")


In [ ]:
# Importar las librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from datetime import datetime

# Importar funciones desde los módulos
from preprocessing import load_subjects_for_experiment, normalize_labels, save_processed_data, clean_memory
from pipeline import compare_pipelines, hold_one_out_experiment, train_and_save_model, plot_subject_performances, plot_confusion_matrices, pipeline_comparison_chart

# Cargar el dataset desde archivos guardados
def load_dataset_from_files(dataset_dir):
    """
    Carga un dataset guardado previamente
    
    Args:
        dataset_dir (str): Ruta al directorio del dataset
        
    Returns:
        dict: Datos cargados
    """
    print(f"Cargando dataset desde: {dataset_dir}")
    
    # Cargar arrays
    X_all = np.load(os.path.join(dataset_dir, 'X_all.npy'))
    y_all = np.load(os.path.join(dataset_dir, 'y_all.npy'))
    subjects_all = np.load(os.path.join(dataset_dir, 'subjects_all.npy'))
    runs_all = np.load(os.path.join(dataset_dir, 'runs_all.npy'))
    
    # Cargar configuración
    with open(os.path.join(dataset_dir, 'dataset_info.json'), 'r') as f:
        config = json.load(f)
    
    # Reconstruir datos en formato de lista de diccionarios para compatibilidad
    eeg_data = []
    
    for subject_info in config['subjects_data']:
        subject_id = subject_info['subject']
        run_id = subject_info['run']
        
        # Identificar índices para este sujeto/run
        start_idx = subject_info['sample_indices']['start']
        end_idx = subject_info['sample_indices']['end']
        
        # Extraer datos para este sujeto
        X_subject = X_all[start_idx:end_idx]
        y_subject = y_all[start_idx:end_idx]
        
        # Crear diccionario con información
        subject_data = {
            'subject': subject_id,
            'run': run_id,
            'task_type': subject_info['task_type'],
            'paradigm': subject_info['paradigm'],
            'experiment_group': subject_info['experiment_group'],
            'X': X_subject,
            'y': y_subject,
            'event_id': {'rest': 1, 'clase1': 2, 'clase2': 3},
            'class_counts': subject_info['class_counts']
        }
        
        eeg_data.append(subject_data)
    
    print(f"Dataset cargado exitosamente: {len(eeg_data)} registros EEG")
    print(f"Grupo de experimento: {config['dataset_info']['experiment_group']}")
    print(f"Total de muestras: {X_all.shape[0]}, Características: {X_all.shape[1]}")
    
    return {
        'eeg_data': eeg_data,
        'config': config,
        'X_all': X_all,
        'y_all': y_all,
        'subjects_all': subjects_all,
        'runs_all': runs_all
    }

# Paso 1: Encontrar el directorio del dataset más reciente
def find_latest_dataset_dir(base_dir='../models'):
    """Encuentra el directorio de dataset más reciente"""
    dataset_dirs = [d for d in os.listdir(base_dir) if d.startswith('eeg_dataset_')]
    if not dataset_dirs:
        raise ValueError(f"No se encontraron datasets en {base_dir}")
    
    # Ordenar por timestamp (parte del nombre)
    latest_dir = sorted(dataset_dirs)[-1]
    return os.path.join(base_dir, latest_dir)

# Paso 2: Cargar el dataset
latest_dataset_dir = find_latest_dataset_dir()
print(f"Usando el dataset más reciente: {latest_dataset_dir}")

dataset = load_dataset_from_files(latest_dataset_dir)
eeg_data = dataset['eeg_data']

# Paso 3: Comparar diferentes pipelines
print("\n=== Comparación de pipelines ===")
pipeline_configs = ['csp_svm', 'freq_rf', 'csp_freq_rf', 'pca_mlp']
comparison_results = compare_pipelines(eeg_data, configs=pipeline_configs)

# Paso 4: Visualizar comparación de pipelines
fig = pipeline_comparison_chart(comparison_results)
plt.savefig(os.path.join('../models', 'pipeline_comparison.png'), dpi=300, bbox_inches='tight')
plt.close(fig)

# Paso 5: Evaluar el mejor pipeline con hold-one-out cross-validation
best_pipeline = comparison_results['best_config']
print(f"\n=== Evaluación del mejor pipeline ({best_pipeline}) con hold-one-out ===")
holdout_results = hold_one_out_experiment(eeg_data, pipeline_config=best_pipeline)

# Paso 6: Visualizar resultados por sujeto
fig_perf = plot_subject_performances(holdout_results)
plt.savefig(os.path.join('../models', 'subject_performances.png'), dpi=300, bbox_inches='tight')
plt.close(fig_perf)

fig_cm = plot_confusion_matrices(holdout_results)
plt.savefig(os.path.join('../models', 'confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.close(fig_cm)

# Paso 7: Entrenar y guardar el modelo final
print("\n=== Entrenamiento del modelo final ===")
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f'eeg_model_{dataset["config"]["dataset_info"]["experiment_group"]}_{timestamp}'
final_model, model_info = train_and_save_model(eeg_data, pipeline_config=best_pipeline, model_name=model_name)

print("\n=== Resumen de la evaluación ===")
print(f"Mejor pipeline: {best_pipeline}")
print(f"Accuracy CV: {comparison_results['results'][best_pipeline]['accuracy']:.4f} ± {comparison_results['results'][best_pipeline]['accuracy_std']:.4f}")
print(f"Hold-one-out Accuracy: {holdout_results['avg_accuracy']:.4f}")
print(f"Modelo final guardado como: {model_info['model_file']}")
print(f"Gráficos de resultados guardados en el directorio '../models'")


In [ ]:
from predict_with_same_experiment import predict_with_same_experiment

# Ejecutar predicción con 6 sujetos nuevos en el mismo tipo de experimento
model_path = '../models/*.joblib'
prediction_results = predict_with_same_experiment(num_subjects=6)
